# Flock Energy Data Science Take-Home Assignment
## REFIT Data Cleaning and Appliance Disaggregation

**Author:** Data Science & Machine Learning Engineering Candidate  
**Objective:** Build a reproducible, well-documented end-to-end pipeline that cleans raw electricity data, explores washing-machine usage, and predicts washing-machine power consumption using whole-house aggregate readings.

---
### Dataset Provenance & Metadata
- **Dataset:** REFIT: Electrical Load Measurements (University of Strathclyde, Loughborough University, University of East Anglia)
- **Funding:** UK EPSRC (Grant Reference EP/K002368/1)
- **Coverage:** 20 UK households in the Loughborough area, October 2013 – June 2015
- **Nominal Sampling Interval:** ~8 seconds (raw), resampled to 1 minute
- **Units:** Active Power in Watts (W)
- **Raw REFIT Data URL:** https://pureportal.strath.ac.uk/files/62090177/REFIT_RAW_081116.7z
- **Official Cleaned REFIT Data URL:** https://pureportal.strath.ac.uk/files/62090184/CLEAN_REFIT_081116.7z
- **Cleaned Readme URL:** https://pure.strath.ac.uk/ws/portalfiles/portal/62090183/CLEAN_READ_ME_081116.txt
- **Dataset Documentation:** https://pureportal.strath.ac.uk/en/datasets/refit-electrical-load-measurements/

## 0. Setup and Dependencies
Import necessary standard libraries: `pandas`, `numpy`, `matplotlib`, `seaborn`, and `scikit-learn`.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, precision_score, recall_score

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10

# Configure project root and paths
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

from config import RAW_DATA_DIR, CLEAN_DATA_DIR, PROCESSED_DATA_DIR, PLOTS_DIR, CLEANING_CONFIG, CYCLE_DETECTION_CONFIG
print(f"Project root configured: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_DIR}")
print(f"Cleaned data path: {CLEAN_DATA_DIR}")

## 1. Raw Data Inspection and Cleaning (House 1)
We load raw electricity data for **House 1** and inspect its structure, timestamps, sampling intervals, and out-of-range sensor readings.

### Documented Cleaning Pipeline:
1. **Timestamp Parsing & Chronological Sorting:** Parse datetime and sort strictly chronologically.
2. **Duplicate Handling Rule:** Multiple readings sharing identical timestamps are aggregated via their arithmetic mean.
3. **Physical Sensor Outlier Handling:**
   - Negative power is physically impossible for consumption monitors; clipped at 0 W.
   - Individual Appliance Monitor (IAM) rated capacity is 4,000 W; any reading > 4,000 W is capped at 4,000 W.
   - Whole-house aggregate service panel limit is 12,000 W (approx. 50A at 240V); spikes above 12,000 W are capped.
4. **1-Minute Resampling:** Resample time-series to 1-minute mean power.
5. **Gap Imputation vs. Outage Flagging:**
   - Short missing gaps ($\le 2$ minutes) are linearly interpolated.
   - Prolonged outages ($\ge 120$ minutes) are explicitly flagged with `is_outage = 1` rather than blind zero-filling or forward-filling.
6. **Visual Inspection:** Plot aggregate power and at least two appliance channels before and after cleaning.

In [ ]:
from clean_raw import RawDataCleaner

raw_file = RAW_DATA_DIR / 'House1.csv'
if not raw_file.exists():
    print(f"[NOTICE] Raw data file not found at {raw_file}.")
    print("Initializing sample verification dataset for testing...")
    from mock_refit_generator import generate_sample_refit_raw_house1
    generate_sample_refit_raw_house1(raw_file, days=7)

cleaner = RawDataCleaner(raw_filepath=raw_file)
df_clean_h1, cleaning_report = cleaner.run_cleaning_pipeline()

print("\n=== House 1 Cleaning Audit Summary ===")
for k, v in cleaning_report.items():
    print(f"{k}: {v}")

### Before and After Cleaning Visualization
Below we display the before-and-after cleaning visualization comparing raw (~8s sampling) against cleaned (1-minute resampled) signals across whole-house aggregate power, Appliance 5 (Washing Machine), and Appliance 1 (Fridge).

In [ ]:
from IPython.display import Image, display
plot_before_after = PLOTS_DIR / 'house1_cleaning_before_after.png'
if plot_before_after.exists():
    display(Image(filename=str(plot_before_after)))
else:
    print("Plot not found. Run cleaning pipeline to generate.")

## 2. Washing-Machine Exploratory Data Analysis
We use official cleaned REFIT data across four households with a washing machine channel: **House 1, House 2, House 3, and House 5**.

### Appliance Metadata Mapping:
| Household | Washing Machine Column | Other Monitored Appliances |
| :--- | :--- | :--- |
| **House 1** | `Appliance5` | Fridge (App1), Chest Freezer (App2), Upright Freezer (App3), Dishwasher (App6) |
| **House 2** | `Appliance2` | Fridge-Freezer (App1), Dishwasher (App3), Television (App4), Microwave (App5) |
| **House 3** | `Appliance5` | Toaster (App1), Fridge-Freezer (App2), Freezer (App3), Dishwasher (App6) |
| **House 5** | `Appliance3` | Fridge-Freezer (App1), Water Heater (App2), Dishwasher (App4), TV (App5) |

### Cycle Detection Rule Formulation:
A domestic washing machine operates through four distinct physical phases:
1. **Water Heating:** Internal resistive heating element draws 2,000–2,400 W for 15–25 minutes.
2. **Drum Tumbling & Agitation:** Motor pulses draw 100–300 W.
3. **Soaking Pauses:** Intermittent pauses where power drops to standby (< 20 W) for up to 10–15 minutes.
4. **High-Speed Spin:** High motor torque draws 500–800 W.

**Our Rule:**
- Active threshold: $P \ge 20\text{ W}$.
- Idle bridging: Gaps $\le 15\text{ minutes}$ below 20 W are bridged to preserve cycle integrity.
- Minimum duration: $\ge 3\text{ minutes}$ to eliminate transient plug-in noise.
- Minimum energy: $\ge 0.05\text{ kWh}$ to confirm an actual wash program.

In [ ]:
from eda_washing_machine import WashingMachineEDA

clean_h1 = CLEAN_DATA_DIR / 'CLEAN_House1.csv'
if not clean_h1.exists():
    print("[NOTICE] Official cleaned files not found. Initializing verification dataset...")
    from mock_refit_generator import generate_sample_clean_households
    generate_sample_clean_households(CLEAN_DATA_DIR, days=14)

eda = WashingMachineEDA()
eda_stats = eda.run_eda(sample_max_days=30)

df_eda = pd.DataFrame(eda_stats).T
df_eda[['Date_Range_Start', 'Date_Range_End', 'Zero_Off_Time_Percent', 'Active_Time_Percent', 'Detected_Cycles_Count', 'Mean_Cycle_Duration_Minutes', 'Mean_Peak_Power_Watts', 'Mean_Energy_Per_Cycle_kWh']]

### Visualizations & Interpretation
1. **Representative 24-Hour Trace:** Demonstrates whole-house aggregate power versus washing machine power during a day with an active wash.
2. **Detected Washing Machine Cycles:** Detailed zoom-in showing the 20 W activation threshold and detected cycle boundaries with duration, peak power, and energy annotations.
3. **Cross-Household Comparison:** Boxplots comparing cycle duration, peak power, and energy per cycle across House 1, House 2, House 3, and House 5.

In [ ]:
# Plot 1: 24h Representative Trace
trace_plot = PLOTS_DIR / 'house1_24h_trace.png'
if trace_plot.exists():
    print("--- 1. Representative 24-Hour Electricity Trace (House 1) ---")
    display(Image(filename=str(trace_plot)))

# Plot 2: Detected Cycles
cycles_plot = PLOTS_DIR / 'house1_detected_cycles.png'
if cycles_plot.exists():
    print("\n--- 2. Detected Washing Machine Operating Cycles (House 1) ---")
    display(Image(filename=str(cycles_plot)))

# Plot 3: Cross-Household Comparison
cross_plot = PLOTS_DIR / 'cross_household_comparison.png'
if cross_plot.exists():
    print("\n--- 3. Cross-Household Comparison of Cycle Metrics ---")
    display(Image(filename=str(cross_plot)))

## 3. Appliance Disaggregation Model (NILM)
We build an appliance disaggregation model to predict washing machine power (Watts) from the whole-house aggregate signal at 1-minute resolution.

### Strict Causal Feature Engineering (Zero Data Leakage):
- Current Aggregate Power: $P_{agg}(t)$
- Backward Lags: $P_{agg}(t-1), P_{agg}(t-2), P_{agg}(t-3), P_{agg}(t-5), P_{agg}(t-10)$
- Step Deltas: $\Delta P_{agg}(t) = P_{agg}(t) - P_{agg}(t-1)$ and $\Delta P_{agg}(t-5)$
- Backward Rolling Statistics: Mean, Standard Deviation, Max, Min over 5, 15, 30, and 60 minutes
- Cyclical Time Features: $\sin / \cos$ transformations of hour-of-day and day-of-week.

### Validation Strategy:
- Chronological train/test split: First 80% for training, last 20% held out for evaluation.
- A 60-minute isolation buffer isolates the training set from the test set to eliminate rolling window overlap.

In [ ]:
from disaggregation_model import DisaggregationPipeline

h1_clean = PROCESSED_DATA_DIR / 'House1_cleaned_1min.csv'
if h1_clean.exists():
    df_h1 = pd.read_csv(h1_clean)
    df_h1['DateTime'] = pd.to_datetime(df_h1['DateTime'])
    if 'Appliance5' in df_h1.columns and 'WashingMachine' not in df_h1.columns:
        df_h1['WashingMachine'] = df_h1['Appliance5']
else:
    df_h1 = eda.load_and_resample_household('House 1', 'Appliance5')

pipeline = DisaggregationPipeline()
df_features = pipeline.engineer_features(df_h1)
train_df, test_df = pipeline.split_data(df_features, split_method='chronological')

results = pipeline.train_and_evaluate(train_df, test_df)
df_results = pd.DataFrame(results).T
print("\n=== Model Evaluation Metrics on Held-Out Test Data ===")
df_results[['Model', 'MAE_Watts', 'RMSE_Watts', 'Energy_Error_Percent', 'Activation_F1_Score', 'Activation_Precision', 'Activation_Recall']]

### Evaluation Metrics Explanation & The Always-Off Trap
- **The Always-Off Baseline Trap:** Because domestic washing machines are inactive >97% of the time, an Always-Off baseline (predicting 0 W constantly) scores an artificially low MAE (~10–15 W)! Relying on MAE alone in NILM gives a false sense of success.
- **Activation F1-Score:** By establishing an activation threshold ($P \ge 20\text{ W}$), we measure the harmonic mean of precision and recall. The always-off baseline scores 0.0, while our machine learning models achieve strong activation F1 scores.
- **Energy Error (\%):** Measures total kilowatt-hour estimation accuracy ($|E_{actual} - E_{pred}| / E_{actual}$), which directly determines the model's reliability for utility demand response and energy efficiency audits.

In [ ]:
# Plot Actual vs Predicted Washing Machine Power
eval_plot_path = pipeline.plot_actual_vs_predicted(test_df, sample_hours=48)
display(Image(filename=str(eval_plot_path)))

### Failure Mode Analysis
Disaggregation algorithms face specific real-world failure modes:
1. **Overlapping High-Power Signatures (False Positives):** Devices with multi-kilowatt resistive heating elements (kettles ~2.5 kW, electric showers ~8 kW, ovens ~2 kW) produce aggregate power surges that resemble washing machine water heating, occasionally triggering false positive predictions.
2. **Low-Power Tumbling/Soak Masking (False Negatives):** During intermediate soak or gentle drum agitation stages (50–150 W), the appliance signature blends into whole-house base loads (refrigerators, entertainment centers), leading to underprediction.

In [ ]:
failure_report = pipeline.analyze_failure_modes(test_df)
print("=== Disaggregation Failure Mode Quantifications ===")
for k, v in failure_report.items():
    print(f"{k}: {v}")

## 4. Practical Implications & Energy-Saving Recommendations

### 1. Customer-Facing Energy-Saving Opportunity
- **Cold-Water Washing Intervention:** 75% to 85% of washing machine energy is consumed heating water from 15°C to 40°C/60°C. Switching from 40°C to 30°C or cold tap temperature saves ~0.4–0.6 kWh per cycle.
- **Utility Measurement:** Utilities can conduct a randomized control trial (RCT). The disaggregation model tracks individual household cycle energy pre- and post-intervention, validating whether treated households achieved sustained reductions compared to control groups.

### 2. Generalization & Transfer Limitations: UK to India
- **Appliance Archetype Differences:** UK washing machines are predominantly 230V front-loading automatic machines with built-in internal water heating elements (~2,200 W). In contrast, Indian urban households predominantly utilize top-loading semi-automatic or cold-fill automatic machines that consume under 300–500 W total (drum motor only, no internal heating element). Models trained on UK data will fail in India without domain adaptation.

### 3. Impact of 15-Minute or 30-Minute Smart-Meter Granularity
- Standard commercial smart meters report readings every 15 or 30 minutes. At this resolution, brief heating spikes (15-20 min) and motor tumbles are smoothed into low-amplitude flat power averages. Edge transitions, cycle duration boundaries, and distinctive load steps become unrecognizable, drastically degrading disaggregation performance.